# Fairness audit of classical ML models on UCI Adult (Group 17)

Same code as `experiments/common.py`, `run_experiments.py` and `run_extensions.py`, in one notebook.
Run all cells from top to bottom (about 3 minutes on CPU). Results are written to `experiments/results/` and figures to `report/figures/` inside the current folder.

In [ ]:
!pip -q install fairlearn

## Shared functions (common.py)

In [ ]:
# things that both experiment scripts need: data loading, split, preprocessing, fairness metrics
import os
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

ROOT = os.getcwd()
FIG = os.path.join(ROOT, "report", "figures")
RES = os.path.join(ROOT, "experiments", "results")
os.makedirs(FIG, exist_ok=True)
os.makedirs(RES, exist_ok=True)

MODELS = {
    "LogReg": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "LinearSVM": LinearSVC(dual="auto", random_state=RANDOM_STATE, max_iter=5000),
    "RandForest": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    "HistGB": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
}


def load_data():
    df = fetch_openml("adult", version=2, as_frame=True).frame.copy()
    text_cols = df.select_dtypes(include=["category", "object"]).columns
    for c in text_cols:
        df[c] = df[c].astype(object).apply(lambda v: v.strip() if isinstance(v, str) else v)
    df[text_cols] = df[text_cols].replace("?", np.nan)

    n_dup = int(df.duplicated().sum())
    df = df.drop_duplicates().reset_index(drop=True)

    df["y"] = df["class"].astype(str).str.contains(">50K").astype(int)
    df["age_group"] = pd.cut(df["age"], [0, 29, 44, 59, 200], labels=["<30", "30-44", "45-59", "60+"]).astype(str)
    return df, n_dup


def split_data(df):
    # 25% test first, then 20% of the rest as validation -> 60/15/25
    y = df["y"]
    s = df[["sex", "race", "age_group"]]
    X = df.drop(columns=["class", "y", "fnlwgt", "age_group"])
    X_rest, X_test, y_rest, y_test, s_rest, s_test = train_test_split(
        X, y, s, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
    X_train, X_val, y_train, y_val, s_train, s_val = train_test_split(
        X_rest, y_rest, s_rest, test_size=0.20, stratify=y_rest, random_state=RANDOM_STATE)
    return X_train, X_val, X_test, y_train, y_val, y_test, s_train, s_val, s_test


def make_preprocessor(X, drop=()):
    num_cols = [c for c in X.select_dtypes(include="number").columns if c not in drop]
    cat_cols = [c for c in X.select_dtypes(exclude="number").columns if c not in drop]
    num_pipe = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())])
    cat_pipe = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                         ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    return ColumnTransformer([("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)])


def get_scores(model, X):
    # the SVM has no predict_proba
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    return model.decision_function(X)


def overall(y_true, y_pred, score=None):
    d = {"accuracy": accuracy_score(y_true, y_pred),
         "f1": f1_score(y_true, y_pred),
         "precision": precision_score(y_true, y_pred),
         "recall": recall_score(y_true, y_pred)}
    if score is not None:
        d["auc"] = roc_auc_score(y_true, score)
    return d


def group_metrics(y_true, y_pred, groups):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    groups = np.asarray(groups)
    rows = []
    for g in pd.unique(groups):
        yt = y_true[groups == g]
        yp = y_pred[groups == g]
        pos = (yt == 1).sum()
        neg = (yt == 0).sum()
        pred_pos = (yp == 1).sum()
        tp = ((yp == 1) & (yt == 1)).sum()
        fp = ((yp == 1) & (yt == 0)).sum()
        fn = ((yp == 0) & (yt == 1)).sum()
        rows.append({"group": g, "n": len(yt),
                     "base_rate": yt.mean(),
                     "selection_rate": yp.mean(),
                     "TPR": tp / pos if pos else np.nan,
                     "FPR": fp / neg if neg else np.nan,
                     "FNR": fn / pos if pos else np.nan,
                     "PPV": tp / pred_pos if pred_pos else np.nan,
                     "accuracy": (yp == yt).mean()})
    return pd.DataFrame(rows).set_index("group")


def gaps(t):
    # gap = biggest group value - smallest group value
    return {"DP": t["selection_rate"].max() - t["selection_rate"].min(),
            "EO": t["TPR"].max() - t["TPR"].min(),
            "FPR": t["FPR"].max() - t["FPR"].min(),
            "FNR": t["FNR"].max() - t["FNR"].min(),
            "PPV": t["PPV"].max() - t["PPV"].min()}


def bootstrap_gaps(y_true, y_pred, groups, n_boot):
    # resample the test set n_boot times and take the 2.5 / 97.5 percentiles of each gap
    rng = np.random.default_rng(RANDOM_STATE)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    groups = np.asarray(groups)
    n = len(y_true)
    out = {"DP": [], "EO": [], "FPR": [], "PPV": []}
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        g = gaps(group_metrics(y_true[idx], y_pred[idx], groups[idx]))
        for k in out:
            out[k].append(g[k])
    ci = {}
    for k in out:
        ci[k] = (float(np.nanpercentile(out[k], 2.5)), float(np.nanpercentile(out[k], 97.5)))
    return ci


def chi2_test(y_pred, groups):
    table = pd.crosstab(np.asarray(groups), np.asarray(y_pred))
    chi2, p, dof, _ = stats.chi2_contingency(table)
    return float(chi2), float(p), int(dof)


def evaluate(model, setting, y_test, y_pred, s_test, score, n_boot):
    # one row of a mitigation table + the per-group tables
    row = {"model": model, "setting": setting}
    row.update(overall(y_test, y_pred, score))
    tables = {}
    for attr in s_test.columns:
        t = group_metrics(y_test, y_pred, s_test[attr])
        tables[attr] = t
        for k, v in gaps(t).items():
            row[attr + "_" + k] = v
    ci = bootstrap_gaps(y_test, y_pred, s_test["sex"], n_boot)
    for k in ci:
        row["sex_" + k + "_lo"] = ci[k][0]
        row["sex_" + k + "_hi"] = ci[k][1]
    print(model, setting, "acc", round(row["accuracy"], 4), "DP", round(row["sex_DP"], 3), "EO", round(row["sex_EO"], 3))
    return row, tables

## Part 1: models, loss curves, audit, reweighing, thresholds

In [ ]:
# Part 1: data, 4 models, loss curves, fairness audit, significance tests, reweighing, group thresholds
# run: .venv/bin/python experiments/run_experiments.py
import json
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss, confusion_matrix


plt.rcParams.update({"font.size": 8, "axes.titlesize": 9, "axes.labelsize": 8,
                     "legend.fontsize": 7, "figure.dpi": 150, "savefig.bbox": "tight"})
summary = {}

In [ ]:
# 1. load and clean
df, n_dup = load_data()
print("rows after cleaning:", len(df), "duplicates dropped:", n_dup)

missing = df.isna().sum()
missing = missing[missing > 0]
summary["data"] = {
    "raw_rows": len(df) + n_dup,
    "duplicates_dropped": n_dup,
    "clean_rows": len(df),
    "missing_per_col": {k: int(v) for k, v in missing.items()},
    "pos_rate": float(df["y"].mean()),
    "sex_counts": df["sex"].value_counts().to_dict(),
    "race_counts": df["race"].value_counts().to_dict(),
    "income_rate_by_sex": df.groupby("sex")["y"].mean().round(4).to_dict(),
    "income_rate_by_race": df.groupby("race")["y"].mean().round(4).to_dict(),
}

# 5 example rows for the report
sample_cols = ["age", "workclass", "education", "marital-status", "occupation", "race", "sex",
               "hours-per-week", "native-country", "class"]
df[sample_cols].sample(5, random_state=RANDOM_STATE).to_csv(os.path.join(RES, "sample_rows.csv"), index=False)

In [ ]:
# 2. split
X_train, X_val, X_test, y_train, y_val, y_test, s_train, s_val, s_test = split_data(df)
summary["split"] = {
    "total": len(df), "train": len(X_train), "val": len(X_val), "test": len(X_test),
    "train_full": len(X_train) + len(X_val),
    "pos_rate": {"train": float(y_train.mean()), "val": float(y_val.mean()), "test": float(y_test.mean())},
    "test_sex_counts": s_test["sex"].value_counts().to_dict(),
    "test_race_counts": s_test["race"].value_counts().to_dict(),
}
print(summary["split"])

In [ ]:
# 3. the fairness functions are in common.py

In [ ]:
# 4. train the 4 models, with and without sex/race, tuned with 3-fold CV on the training set
GRIDS = {
    "LogReg": {"clf__C": [0.01, 0.1, 1, 10]},
    "LinearSVM": {"clf__C": [0.01, 0.1, 1]},
    "RandForest": {"clf__n_estimators": [200], "clf__max_depth": [10, 20, None], "clf__min_samples_leaf": [1, 5]},
    "HistGB": {"clf__learning_rate": [0.05, 0.1], "clf__max_iter": [200, 400], "clf__max_leaf_nodes": [15, 31]},
}
DROP = {"with": (), "without": ("sex", "race")}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

results = []
per_group = []
preds, scores, fitted, best_params = {}, {}, {}, {}
for name in MODELS:
    for variant in ["with", "without"]:
        pipe = Pipeline([("pre", make_preprocessor(X_train, DROP[variant])), ("clf", MODELS[name])])
        gs = GridSearchCV(pipe, GRIDS[name], cv=cv, scoring="f1", n_jobs=-1)
        start = time.time()
        gs.fit(X_train, y_train)
        fit_time = time.time() - start

        model = gs.best_estimator_
        y_pred_val = model.predict(X_val)
        y_pred = model.predict(X_test)
        score = get_scores(model, X_test)
        preds[(name, variant)] = y_pred
        scores[(name, variant)] = score
        fitted[(name, variant)] = model
        best_params[name + "_" + variant] = {k.replace("clf__", ""): v for k, v in gs.best_params_.items()}

        row = {"model": name, "variant": variant, "cv_f1": gs.best_score_,
               "val_accuracy": accuracy_score(y_val, y_pred_val), "val_f1": f1_score(y_val, y_pred_val),
               "fit_time_s": fit_time}
        row.update(overall(y_test, y_pred, score))
        for attr in ["sex", "race"]:
            t = group_metrics(y_test, y_pred, s_test[attr])
            for k, v in gaps(t).items():
                row[attr + "_" + k] = v
            t = t.reset_index()
            t.insert(0, "attr", attr)
            t.insert(0, "variant", variant)
            t.insert(0, "model", name)
            per_group.append(t)
        results.append(row)
        print(name, variant, gs.best_params_, "acc", round(row["accuracy"], 4), "f1", round(row["f1"], 4),
              "sex DP", round(row["sex_DP"], 4), "sex EO", round(row["sex_EO"], 4))

res_df = pd.DataFrame(results)
res_df.to_csv(os.path.join(RES, "model_results.csv"), index=False)
pd.concat(per_group).to_csv(os.path.join(RES, "per_group_metrics.csv"), index=False)
summary["best_params"] = best_params

In [ ]:
# 5. loss curves
# logistic regression with SGD, one pass over the training data = one epoch
pre = make_preprocessor(X_train).fit(X_train)
Xtr = pre.transform(X_train)
Xva = pre.transform(X_val)
sgd = SGDClassifier(loss="log_loss", alpha=1e-4, learning_rate="optimal", random_state=RANDOM_STATE)
EPOCHS = 40
tr_loss, va_loss, va_acc = [], [], []
rng = np.random.default_rng(RANDOM_STATE)
for ep in range(EPOCHS):
    order = rng.permutation(len(Xtr))
    sgd.partial_fit(Xtr[order], y_train.values[order], classes=[0, 1])
    tr_loss.append(log_loss(y_train, sgd.predict_proba(Xtr)[:, 1]))
    va_loss.append(log_loss(y_val, sgd.predict_proba(Xva)[:, 1]))
    va_acc.append(accuracy_score(y_val, sgd.predict(Xva)))
summary["sgd_curve"] = {"epochs": EPOCHS, "train_loss_first": tr_loss[0], "train_loss_last": tr_loss[-1],
                        "val_loss_first": va_loss[0], "val_loss_last": va_loss[-1],
                        "val_loss_min": float(np.min(va_loss)), "val_loss_argmin": int(np.argmin(va_loss)) + 1,
                        "val_acc_last": va_acc[-1]}

# gradient boosting, loss after every boosting iteration
hgb = HistGradientBoostingClassifier(learning_rate=0.1, max_iter=400, max_leaf_nodes=31,
                                     early_stopping=False, random_state=RANDOM_STATE)
hgb.fit(Xtr, y_train)
g_tr = [log_loss(y_train, p[:, 1]) for p in hgb.staged_predict_proba(Xtr)]
g_va = [log_loss(y_val, p[:, 1]) for p in hgb.staged_predict_proba(Xva)]
summary["hgb_curve"] = {"iters": len(g_tr), "train_first": g_tr[0], "train_last": g_tr[-1],
                        "val_first": g_va[0], "val_last": g_va[-1],
                        "val_min": float(np.min(g_va)), "val_argmin": int(np.argmin(g_va)) + 1}
print("sgd", summary["sgd_curve"])
print("hgb", summary["hgb_curve"])

epochs = np.arange(1, EPOCHS + 1)
iters = np.arange(1, len(g_tr) + 1)
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.3))
axes[0].plot(epochs, tr_loss, color="#2563eb", label="training loss")
axes[0].plot(epochs, va_loss, color="#ea580c", ls="--", label="validation loss")
axes[0].set_xlabel("epoch (one SGD pass over the training set)")
axes[0].set_title("(a) Logistic regression trained by SGD")
axes[1].plot(iters, g_tr, color="#2563eb", label="training loss")
axes[1].plot(iters, g_va, color="#ea580c", ls="--", label="validation loss")
axes[1].axvline(summary["hgb_curve"]["val_argmin"], color="#6b7280", lw=0.8, ls=":")
axes[1].set_xlabel("boosting iteration")
axes[1].set_title("(b) Gradient boosting")
for ax in axes:
    ax.set_ylabel("log loss")
    ax.legend()
    ax.grid(alpha=.3)
fig.savefig(os.path.join(FIG, "loss_curve.pdf"))
plt.close(fig)

pd.DataFrame({"epoch": epochs, "train_loss": tr_loss, "val_loss": va_loss, "val_acc": va_acc}).to_csv(
    os.path.join(RES, "sgd_loss_curve.csv"), index=False)
pd.DataFrame({"iter": iters, "train_loss": g_tr, "val_loss": g_va}).to_csv(
    os.path.join(RES, "hgb_loss_curve.csv"), index=False)

In [ ]:
# 6. are the gaps significant? bootstrap CI, z-test and chi-square test for every model
def ztest(y_pred, groups, a, b):
    # two-proportion z-test on the selection rates of group a and group b
    y_pred = np.asarray(y_pred)
    groups = np.asarray(groups)
    xa, na = y_pred[groups == a].sum(), (groups == a).sum()
    xb, nb = y_pred[groups == b].sum(), (groups == b).sum()
    p = (xa + xb) / (na + nb)
    z = (xa / na - xb / nb) / np.sqrt(p * (1 - p) * (1 / na + 1 / nb))
    return float(z), float(2 * stats.norm.sf(abs(z)))

stat_rows = []
for (name, variant), y_pred in preds.items():
    for attr in ["sex", "race"]:
        chi2, chi2_p, dof = chi2_test(y_pred, s_test[attr])
        row = {"model": name, "variant": variant, "attr": attr, "chi2": chi2, "chi2_p": chi2_p, "dof": dof}
        ci = bootstrap_gaps(y_test, y_pred, s_test[attr], 1000)
        for k in ci:
            row[k + "_lo"] = ci[k][0]
            row[k + "_hi"] = ci[k][1]
        if attr == "sex":
            row["z"], row["z_p"] = ztest(y_pred, s_test[attr], "Male", "Female")
        else:
            row["z"], row["z_p"] = ztest(y_pred, s_test[attr], "White", "Black")
        stat_rows.append(row)
pd.DataFrame(stat_rows).to_csv(os.path.join(RES, "significance.csv"), index=False)
print("significance tests done")

In [ ]:
# 7. mitigation for sex on gradient boosting and logistic regression (starting from the "without" model)
def reweigh_weights(groups, labels):
    # Kamiran & Calders: w(g, y) = P(g) * P(y) / P(g, y)
    groups = np.asarray(groups)
    labels = np.asarray(labels)
    w = np.ones(len(labels))
    for g in np.unique(groups):
        for l in np.unique(labels):
            m = (groups == g) & (labels == l)
            w[m] = (groups == g).mean() * (labels == l).mean() / m.mean()
    return w

def group_thresholds(score, y, groups):
    # one threshold per group so that each group gets the same TPR as the whole validation set has at 0.5
    score = np.asarray(score)
    y = np.asarray(y)
    groups = np.asarray(groups)
    target_tpr = ((score >= 0.5) & (y == 1)).sum() / (y == 1).sum()
    th = {}
    for g in np.unique(groups):
        pos_scores = score[(groups == g) & (y == 1)]
        th[g] = float(np.quantile(pos_scores, 1 - target_tpr))
    return th, float(target_tpr)

def apply_thresholds(score, groups, th):
    return np.array([int(s >= th[g]) for s, g in zip(score, groups)])

mit_rows = []
mit_tables = []
def record(name, setting, y_pred, score):
    row, tables = evaluate(name, setting, y_test, y_pred, s_test[["sex", "race"]], score, 500)
    mit_rows.append(row)
    for attr in tables:
        mit_tables.append(tables[attr].reset_index().assign(model=name, setting=setting, attr=attr))

w = reweigh_weights(s_train["sex"], y_train)
for name in ["HistGB", "LogReg"]:
    base = fitted[(name, "without")]
    base_score = scores[(name, "without")]
    record(name, "with sex/race", preds[(name, "with")], scores[(name, "with")])
    record(name, "without sex/race", preds[(name, "without")], base_score)

    # reweighing: same model and hyperparameters, trained with sample weights
    est = clone(MODELS[name]).set_params(**best_params[name + "_without"])
    rw = Pipeline([("pre", make_preprocessor(X_train, ("sex", "race"))), ("clf", est)])
    rw.fit(X_train, y_train, clf__sample_weight=w)
    rw_score = rw.predict_proba(X_test)[:, 1]
    record(name, "+ reweighing", rw.predict(X_test), rw_score)
    if name == "HistGB":
        rw_hgb = rw

    # group thresholds, picked on the validation set
    th, target = group_thresholds(base.predict_proba(X_val)[:, 1], y_val, s_val["sex"])
    record(name, "+ group thresholds", apply_thresholds(base_score, s_test["sex"], th), base_score)
    summary["thresholds_" + name] = {"thresholds": th, "target_tpr": target}

    # both together
    th2, _ = group_thresholds(rw.predict_proba(X_val)[:, 1], y_val, s_val["sex"])
    record(name, "+ reweighing + thresholds", apply_thresholds(rw_score, s_test["sex"], th2), rw_score)

    weights = {}
    for g in ["Male", "Female"]:
        for l in [0, 1]:
            weights[g + "_" + str(l)] = float(w[(s_train["sex"].values == g) & (y_train.values == l)][0])
    summary["reweigh_weights_" + name] = weights

pd.DataFrame(mit_rows).to_csv(os.path.join(RES, "mitigation.csv"), index=False)
pd.concat(mit_tables).to_csv(os.path.join(RES, "mitigation_per_group.csv"), index=False)

# confusion matrix of the best model (by F1)
best = res_df.sort_values("f1", ascending=False).iloc[0]
cm = confusion_matrix(y_test, preds[(best["model"], best["variant"])])
summary["best_model"] = {"model": best["model"], "variant": best["variant"], "cm": cm.tolist()}
print("best model:", best["model"], best["variant"], cm.tolist())

# save the gradient boosting models for the demo
cat_cols = X_train.select_dtypes(exclude="number").columns
demo_models = {
    "with": fitted[("HistGB", "with")],
    "without": fitted[("HistGB", "without")],
    "reweighed": rw_hgb,
    "columns": X_train.columns.tolist(),
    "example": X_test.iloc[:200].reset_index(drop=True),
    "example_y": y_test.iloc[:200].reset_index(drop=True),
    "choices": {c: sorted(df[c].dropna().unique().tolist()) for c in cat_cols},
}
os.makedirs(os.path.join(ROOT, "demo"), exist_ok=True)
joblib.dump(demo_models, os.path.join(ROOT, "demo", "models.joblib"))

with open(os.path.join(RES, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2, default=str)
print("done, results are in", RES)

## Part 2: age groups, sex x race, more mitigation methods

In [ ]:
# Part 2: age groups, PPV gap, sex x race groups, and three more mitigation methods
# (disparate impact remover, equalized odds post-processing, exponentiated gradient)
# run after run_experiments.py: .venv/bin/python experiments/run_extensions.py
import json
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.reductions import ExponentiatedGradient, DemographicParity, EqualizedOdds


plt.rcParams.update({"font.size": 7, "axes.titlesize": 7.5, "axes.labelsize": 7,
                     "legend.fontsize": 6, "figure.dpi": 150, "savefig.bbox": "tight"})
ext = {}

In [ ]:
# same data and same split as part 1
df, n_dup = load_data()
X_train, X_val, X_test, y_train, y_val, y_test, s_train, s_val, s_test = split_data(df)
sex_race = s_test["sex"] + "/" + s_test["race"]
ext["age_group_counts_test"] = s_test["age_group"].value_counts().to_dict()
ext["age_income_rate"] = df.groupby("age_group")["y"].mean().round(4).to_dict()
ext["intersectional_counts_test"] = sex_race.value_counts().to_dict()
print(ext["age_group_counts_test"])

In [ ]:
# A. train the 8 models again with the hyperparameters found in part 1, audit by sex, race and age group
with open(os.path.join(RES, "summary.json")) as f:
    best_params = json.load(f)["best_params"]

rows = []
per_group = []
preds, scores, fitted = {}, {}, {}
for name in MODELS:
    for variant in ["with", "without"]:
        drop = () if variant == "with" else ("sex", "race")
        est = clone(MODELS[name]).set_params(**best_params[name + "_" + variant])
        model = Pipeline([("pre", make_preprocessor(X_train, drop)), ("clf", est)]).fit(X_train, y_train)
        y_pred = model.predict(X_test)
        fitted[(name, variant)] = model
        preds[(name, variant)] = y_pred
        scores[(name, variant)] = get_scores(model, X_test)

        row = {"model": name, "variant": variant, "accuracy": accuracy_score(y_test, y_pred)}
        for attr in ["sex", "race", "age_group"]:
            t = group_metrics(y_test, y_pred, s_test[attr])
            for k, v in gaps(t).items():
                row[attr + "_" + k] = v
            chi2, p, dof = chi2_test(y_pred, s_test[attr])
            row[attr + "_chi2"] = chi2
            row[attr + "_chi2_p"] = p
            t = t.reset_index()
            t.insert(0, "attr", attr)
            t.insert(0, "variant", variant)
            t.insert(0, "model", name)
            per_group.append(t)
        ci = bootstrap_gaps(y_test, y_pred, s_test["age_group"], 300)
        row["age_DP_lo"], row["age_DP_hi"] = ci["DP"]
        row["age_EO_lo"], row["age_EO_hi"] = ci["EO"]
        rows.append(row)
        print(name, variant, "acc", round(row["accuracy"], 4), "age DP", round(row["age_group_DP"], 3),
              "age EO", round(row["age_group_EO"], 3), "sex PPV gap", round(row["sex_PPV"], 3))

ext_df = pd.DataFrame(rows)
ext_df.to_csv(os.path.join(RES, "ext_model_results.csv"), index=False)
pd.concat(per_group).to_csv(os.path.join(RES, "ext_per_group_metrics.csv"), index=False)

# check that the retrained models give the same accuracy as in part 1
part1 = pd.read_csv(os.path.join(RES, "model_results.csv"))
assert np.allclose(ext_df["accuracy"], part1["accuracy"]), "models do not match part 1"

In [ ]:
# B. sex x race groups for gradient boosting
inter = []
for variant in ["with", "without"]:
    t = group_metrics(y_test, preds[("HistGB", variant)], sex_race).reset_index()
    t.insert(0, "variant", variant)
    inter.append(t)
inter = pd.concat(inter)
inter.to_csv(os.path.join(RES, "ext_intersectional.csv"), index=False)

# only groups with at least 100 test rows
big = inter[(inter["variant"] == "with") & (inter["n"] >= 100)]
ext["intersectional_with_n>=100"] = {
    "DP": float(big["selection_rate"].max() - big["selection_rate"].min()),
    "EO": float(big["TPR"].max() - big["TPR"].min()),
    "max_group": big.loc[big["selection_rate"].idxmax(), "group"],
    "min_group": big.loc[big["selection_rate"].idxmin(), "group"],
}
print("intersectional", ext["intersectional_with_n>=100"])

In [ ]:
# C. more mitigation methods (for sex)
class DisparateImpactRemover:
    # Feldman et al. 2015, numeric columns only.
    # A value at quantile q inside its own group is replaced by the value at quantile q
    # of the "median distribution" (median of the group quantiles).
    def __init__(self, cols, repair_level=1.0):
        self.cols = cols
        self.level = repair_level
        self.q = np.linspace(0, 1, 101)

    def fit(self, X, groups):
        groups = np.asarray(groups)
        self.groups = np.unique(groups)
        self.group_q = {}
        self.median_q = {}
        for c in self.cols:
            v = X[c].astype(float).values
            self.group_q[c] = {g: np.nanquantile(v[groups == g], self.q) for g in self.groups}
            self.median_q[c] = np.median(np.vstack(list(self.group_q[c].values())), axis=0)
        return self

    def transform(self, X, groups):
        X = X.copy()
        groups = np.asarray(groups)
        for c in self.cols:
            v = X[c].astype(float).values
            out = v.copy()
            for g in self.groups:
                m = (groups == g) & ~np.isnan(v)
                if not m.any():
                    continue
                rank = np.searchsorted(self.group_q[c][g], v[m], side="right") / len(self.q)
                rank = np.clip(rank, 0, 1)
                repaired = np.interp(rank, self.q, self.median_q[c])
                out[m] = (1 - self.level) * v[m] + self.level * repaired
            X[c] = out
        return X

mit_rows = []
mit_tables = []
def record(name, setting, y_pred, score):
    row, tables = evaluate(name, setting, y_test, y_pred, s_test, score, 300)
    mit_rows.append(row)
    for attr in tables:
        mit_tables.append(tables[attr].reset_index().assign(model=name, setting=setting, attr=attr))

num_cols = X_train.select_dtypes(include="number").columns.tolist()
for name in ["HistGB", "LogReg"]:
    base = fitted[(name, "without")]
    base_est = base.named_steps["clf"]
    record(name, "without sex/race", preds[(name, "without")], scores[(name, "without")])

    # disparate impact remover: repair the numeric columns, then train again
    dir_ = DisparateImpactRemover(num_cols).fit(X_train, s_train["sex"])
    X_train_rep = dir_.transform(X_train, s_train["sex"])
    X_test_rep = dir_.transform(X_test, s_test["sex"])
    model = Pipeline([("pre", make_preprocessor(X_train, ("sex", "race"))), ("clf", clone(base_est))])
    model.fit(X_train_rep, y_train)
    record(name, "+ DI remover (lambda=1)", model.predict(X_test_rep), model.predict_proba(X_test_rep)[:, 1])

    # equalized odds post-processing (Hardt et al.), fitted on the validation set
    to = ThresholdOptimizer(estimator=base, constraints="equalized_odds", objective="accuracy_score",
                            prefit=True, predict_method="predict_proba")
    to.fit(X_val, y_val, sensitive_features=s_val["sex"])
    y_pred = to.predict(X_test, sensitive_features=s_test["sex"], random_state=RANDOM_STATE)
    record(name, "+ EO post-processing (Hardt)", y_pred, scores[(name, "without")])

    # in-processing: exponentiated gradient (Agarwal et al.) with a DP constraint and with an EO constraint
    pre = make_preprocessor(X_train, ("sex", "race")).fit(X_train)
    Ztr = pre.transform(X_train)
    Zte = pre.transform(X_test)
    constraints = {"DP": DemographicParity(difference_bound=0.01), "EO": EqualizedOdds(difference_bound=0.01)}
    for cname in constraints:
        max_iter = 50 if name == "LogReg" else 25
        eg = ExponentiatedGradient(clone(base_est), constraints=constraints[cname], eps=0.01, max_iter=max_iter)
        eg.fit(Ztr, y_train, sensitive_features=s_train["sex"])
        y_pred = np.asarray(eg.predict(Zte, random_state=RANDOM_STATE))
        score = eg._pmf_predict(Zte)[:, 1]
        record(name, "+ in-processing EG (" + cname + ")", y_pred, score)
        print("   classifiers in the ensemble:", len(eg.predictors_))

mit_df = pd.DataFrame(mit_rows)
mit_df.to_csv(os.path.join(RES, "ext_mitigation.csv"), index=False)
pd.concat(mit_tables).to_csv(os.path.join(RES, "ext_mitigation_per_group.csv"), index=False)

In [ ]:
# figure: gradient boosting by age group
t = group_metrics(y_test, preds[("HistGB", "with")], s_test["age_group"]).loc[["<30", "30-44", "45-59", "60+"]]
fig, ax = plt.subplots(figsize=(3.4, 1.9))
t[["base_rate", "selection_rate", "TPR", "FPR"]].plot.bar(
    ax=ax, rot=0, color=["#6b7280", "#2563eb", "#16a34a", "#ea580c"], width=0.8)
ax.set_ylabel("rate")
ax.set_xlabel("age group")
ax.legend(["true >50K rate", "selection rate", "TPR", "FPR"], ncol=2)
ax.grid(axis="y", alpha=.3)
ax.set_title("Gradient boosting by age group (test set)")
fig.savefig(os.path.join(FIG, "age_groups.pdf"))
plt.close(fig)

# figure: accuracy vs gap for every mitigation setting (part 1 and part 2 together)
old = pd.read_csv(os.path.join(RES, "mitigation.csv"))
allm = pd.concat([old[old["setting"] != "without sex/race"], mit_df]).reset_index(drop=True)
colors = {"with sex/race": "#6b7280", "without sex/race": "#2563eb", "+ reweighing": "#dc2626",
          "+ group thresholds": "#16a34a", "+ reweighing + thresholds": "#7c3aed",
          "+ DI remover (lambda=1)": "#92400e", "+ EO post-processing (Hardt)": "#0d9488",
          "+ in-processing EG (DP)": "#be185d", "+ in-processing EG (EO)": "#eab308"}
markers = {"HistGB": "D", "LogReg": "o"}
fig, axes = plt.subplots(1, 2, figsize=(3.4, 2.0))
for ax, col, label in zip(axes, ["sex_DP", "sex_EO"], ["DP gap (sex)", "EO gap (sex)"]):
    for _, r in allm.iterrows():
        ax.scatter(r[col], r["accuracy"], marker=markers[r["model"]], s=24, color=colors[r["setting"]],
                   edgecolor="k", lw=0.3)
    ax.set_xlabel(label)
    ax.grid(alpha=.3)
    ax.tick_params(labelsize=6)
axes[0].set_ylabel("test accuracy")
handles = [Line2D([], [], marker="D", color="k", ls="", ms=4, label="Grad. boosting"),
           Line2D([], [], marker="o", color="k", ls="", ms=4, label="LogReg")]
for setting in colors:
    label = setting.replace("(lambda=1)", "").replace("in-processing ", "")
    handles.append(Line2D([], [], marker="o", color=colors[setting], ls="", ms=5, label=label))
fig.tight_layout()
fig.legend(handles=handles, loc="lower center", ncol=3, fontsize=5.2, frameon=False, bbox_to_anchor=(0.5, -0.27))
fig.savefig(os.path.join(FIG, "tradeoff_ext.pdf"))
plt.close(fig)

with open(os.path.join(RES, "summary_ext.json"), "w") as f:
    json.dump(ext, f, indent=2, default=str)
print("done")